# VaxGenesis AI — Platform Walkthrough

This notebook demonstrates the VaxGenesis AI platform end-to-end, from genome upload to digital vaccine twin generation. It is intended for:

- **New engineers** onboarding to the codebase
- **Scientists** evaluating the platform's capabilities
- **Stakeholders** reviewing the v1.2 hexagonal architecture

**Prerequisites:**
- The backend is running at `http://localhost:8000` (or set `BACKEND_URL` below)
- Python 3.11+ with `httpx`, `pydantic`, `rich` installed

**Sections:**
1. Authentication
2. Project + pathogen creation
3. Candidate creation and pipeline run
4. Surveillance: variant ingestion and escape detection
5. Self-learning: active learning queue and assay submission
6. Hexagonal architecture demo (v2 API)

In [ ]:
# Setup
import httpx
import json
from datetime import datetime
from rich.console import Console
from rich.table import Table
from rich.json import JSON

console = Console()
BACKEND_URL = "http://localhost:8000"
client = httpx.Client(base_url=BACKEND_URL, timeout=30.0)

def show(title, data):
    console.print(f"\n[bold cyan]{title}[/bold cyan]")
    if isinstance(data, (dict, list)):
        console.print(JSON.from_data(data))
    else:
        console.print(data)

console.print("[green]✓[/green] Notebook initialized. Backend: " + BACKEND_URL)

## 1. Authentication

Login with the default admin credentials to obtain a JWT token.

In [ ]:
response = client.post("/api/v1/auth/login", json={
    "email": "admin@vaxgenesis.ai",
    "password": "admin123",
})
assert response.status_code == 200, f"Login failed: {response.text}"
token = response.json()["access_token"]
client.headers["Authorization"] = f"Bearer {token}"
show("Authentication", {"status": "success", "token": token[:20] + "..."})

## 2. Project + Pathogen Creation

Create a new vaccine design project and upload a reference genome.

In [ ]:
# Create a project
project = client.post("/api/v1/projects", json={
    "name": f"Notebook Demo {datetime.now().strftime('%Y-%m-%d %H:%M')}",
    "description": "Created from the Jupyter notebook walkthrough",
}).json()
show("Project Created", project)
project_id = project["id"]

In [ ]:
# Upload a reference genome (synthetic SARS-CoV-2-like fragment)
reference_genome = (
    "ATGGCTAGCAAACTGGAACGTCTGGTACCAACTGAAAGTGCTGGCCAAGCTGGTGAAAGTG"
    "ACCCGGCTGAAAGTGGAACGTCTGACCGGTAAACTGAAAGTGAAACTGACCGGTACCAAA"
    "CTGAAAGTGCTGACCAAACTGGTGAAGTGGCTAAACTGACCAAACTGAAAGTGACCAAAT"
    "AA"
)
pathogen = client.post("/api/v1/pathogens", json={
    "project_id": project_id,
    "name": "Notebook Demo Virus",
    "type": "virus",
    "reference_genome": reference_genome,
}).json()
show("Pathogen Uploaded", {"id": pathogen["id"], "name": pathogen["name"], "gc_content": pathogen.get("gc_content")})

## 3. Candidate Creation and Pipeline Run

Create a vaccine candidate and trigger the autonomous reasoning pipeline.

In [ ]:
candidate = client.post("/api/v1/candidates", json={
    "project_id": project_id,
    "name": "Notebook Candidate v1",
    "platform": "multi_epitope",
}).json()
show("Candidate Created", candidate)
candidate_id = candidate["id"]

In [ ]:
# View the modules catalog (should show 31 modules: 27 v1.0 + 4 v1.1 audit additions)
modules = client.get("/api/v1/modules").json()
console.print(f"\n[bold cyan]Module Catalog ({len(modules)} modules)[/bold cyan]")
table = Table(show_header=True, header_style="bold magenta")
table.add_column("#")
table.add_column("ID")
table.add_column("Name")
table.add_column("Category")
for i, m in enumerate(modules, 1):
    table.add_row(str(i), m["id"], m["name"], m["category"])
console.print(table)

## 4. Surveillance: Variant Ingestion

Ingest a synthetic variant sequence and check the surveillance status.

In [ ]:
# Check surveillance status
status = client.get("/api/v1/surveillance/status").json()
show("Surveillance Status (before)", status)

## 5. Hexagonal Architecture Demo (v2 API)

The v2 API exposes the same capabilities through a hexagonal architecture
with explicit ports, adapters, and use cases. This is the future direction
of the platform (see `docs/adr/0001-hexagonal-architecture.md`).

In [ ]:
# Use the v2 hexagonal API to create a candidate
# Note: this requires the v2 router to be mounted on the backend
v2_response = client.post("/api/v2/candidates", json={
    "project_id": project_id,
    "name": "Hexagonal v2 Candidate",
    "platform": "multi_epitope",
})
if v2_response.status_code == 201:
    show("v2 Candidate Created (Hexagonal)", v2_response.json())
else:
    show("v2 API Status", {"status_code": v2_response.status_code, "note": "v2 router may not be mounted yet"})

## 6. Domain Layer Demo (Direct Python)

The hexagonal domain layer can be used directly in Python (useful for
notebooks, scripts, and tests) without going through the REST API.

In [ ]:
# Import the hexagonal domain layer directly
import sys
sys.path.insert(0, "../apps/backend/src")

from vaxgenesis.hexagon.domain.entities import Candidate, CandidateStatus, VaccinePlatform
from vaxgenesis.hexagon.domain.value_objects import (
    Sequence, ConfidenceInterval, UncertaintyScore, ProjectId, HlaAllele
)
from vaxgenesis.hexagon.domain.errors import InvalidStateTransitionError

# Create a domain entity
candidate = Candidate(
    project_id=ProjectId(),
    name="Domain Demo Candidate",
    platform=VaccinePlatform.MULTI_EPITOPE,
)
show("Domain Candidate (initial)", {"name": candidate.name, "status": candidate.status, "version": candidate.version})

In [ ]:
# Demonstrate the state machine
try:
    # This should fail — can't go to VALIDATED without PREDICTED first
    candidate.advance_status(CandidateStatus.VALIDATED)
except InvalidStateTransitionError as e:
    show("State Machine Enforcement", {"error": str(e), "note": "Correctly rejected invalid transition"})

In [ ]:
# Valid transition: set predictions, then advance
candidate.set_predictions(
    overall_score=0.82,
    confidence_interval=ConfidenceInterval(low=0.72, high=0.92),
    uncertainty=UncertaintyScore(value=0.25, method="bayesian"),
)
candidate.advance_status(CandidateStatus.PREDICTED)
show("After Prediction", {
    "status": candidate.status,
    "score": candidate.overall_score,
    "uncertainty": candidate.uncertainty.value,
    "ci": [candidate.confidence_interval.low, candidate.confidence_interval.high],
})

In [ ]:
# Demonstrate value object validation
seq = Sequence(value="ATGGCTAAAGGCCGG", alphabet="dna")
hla = HlaAllele(value="HLA-A*02:01")
show("Value Objects", {
    "sequence": {"value": seq.value, "length": seq.length, "gc_content": round(seq.gc_content(), 3)},
    "hla_allele": {"value": hla.value, "locus": hla.locus, "mhc_class": hla.mhc_class},
})

## Summary

This notebook demonstrated:

1. **v1 REST API** — project, pathogen, candidate, surveillance endpoints
2. **v2 Hexagonal API** — same capabilities through ports & adapters
3. **Direct domain layer usage** — entities, value objects, state machines

### Key takeaways

- The domain layer is pure Python (Pydantic v2) with zero infrastructure dependencies.
- The state machine enforces invariants (e.g., can't validate without predictions).
- Value objects validate at construction time (invalid sequences never exist).
- The v2 API is a thin wrapper over use cases; no business logic in routes.

### Next steps

- Read the ADRs in `docs/adr/` for architectural rationale.
- Run the property-based tests: `pytest tests/property/ -v`
- Run the benchmarks: `pytest tests/benchmark/ -v --benchmark-only`
- Read `docs/MIGRATION_v1_to_v2.md` for the migration guide.